# Suite No. 3 — Scale-Dependent Low-Frequency Structure in Proxy Field Readouts

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HNXJ/jaxfne/blob/main/tutorials/jaxfne_suite_no_3_low_frequency_scaling.ipynb)

**Tutorial type:** `jaxfne` / TFNE tutorial atlas notebook  
**Execution target:** Antigravity or Colab-style notebook runner  
**Canonical import:** `import jaxfne as jtfne`  
**Truth status:** `truth_safe_unverified` / `computational_scaffold`  
**Readout status:** simulated/proxy readouts; no calibrated physical amplitude interpretation  

This suite demonstrates scale-dependent low-frequency scaling behavior in noisy asynchronous-irregular spiking populations under controlled spatiotemporal density. It uses whole-window absolute power spectrum estimators to analyze 1/f^alpha power-law scaling versus population scale.

## 1. Learning objectives

By the end of this suite, you will:

- Configure scale-dependent populations while holding spatiotemporal density constant
- Setup heterogeneous drives to establish a stable asynchronous-irregular spiking regime
- Compute whole-window absolute power spectra on log-log axes
- Fit 1/f^alpha scaling exponents in the 1-80 Hz frequency band
- Validate scale-dependent metrics (slope alpha, low-frequency power, synchrony) across population sizes

## 2. Biological/computational question

**Question:** How does scaling the population size N while preserving constant spatiotemporal density alter the absolute power-law exponent in aggregate field readouts?

**Context:** 
In a noisy asynchronous-irregular regime, independent fluctuations average out under projection, leaving low-frequency modes to scale with population size. Ensuring constant density prevents confounding local packaging density with population scale.

## 3. Mathematical glossary flow

### Formal equation

$$Y_c(t) = \sum_{n=1}^{N} W_{cn} S_n(t)$$

where $Y_c(t)$ is the proxy readout at contact $c$, $W_{cn}$ is the projection weight, and $S_n(t)$ is the source current.

### Terms

| Symbol | Meaning | Status |
|---|---|---|
| $Y_c(t)$ | proxy readout |
| $W_{cn}$ | projection weight |
| $S_n(t)$ | source current |

### Whole-Window Absolute Power Spectrum

$$P(f) = \frac{|Y(f)|^2}{\text{normalization}}$$

where $Y(f)$ is the Fourier transform of the windowed, detrended signal.

### Log-Log Power-Law Fit

$$\log_{10}(P(f)) = \beta_0 - \alpha \log_{10}(f)$$

where $\alpha$ represents the power-law exponent in the 1-80 Hz band.

### Scope boundary

This is a proxy source-to-readout projection unless a run supplies physical geometry, calibrated source units, conductivity, boundary conditions, gauge handling, a physical field solver, and validation evidence.

In [1]:
import jaxfne as jtfne
import jax.numpy as jnp
import jax
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path
from dataclasses import replace

print(f"jaxfne version: {jtfne.__version__}")

jaxfne version: 0.3.5


## 4. Scaling Experiment Parameters

Declare parameters for the constant-density scaling experiment.

In [2]:
SEED = 303
DURATION_MS = 1000.0
DT_MS = 0.1
DTYPE = "float32"

SCALES = [10, 50, 100, 500]
E_FRACTION = 0.75
I_FRACTION = 0.25
LAYER_NAMES = ["L2/3"]
PROBE_MODES = ["spk", "vm", "source", "lfp_proxy"]

FIT_BAND = (1.0, 80.0)
SYNCHRONY_BIN_MS = 10.0

OUTPUT_DIR = Path("outputs") / "suite_no3_low_frequency_scaling"
FIG_DIR = OUTPUT_DIR / "figures"
for path in [OUTPUT_DIR, FIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# Public metadata structure for validation contract
RUN_METADATA = {
    "truth_mode": "truth_safe_unverified",
    "claim_level": "computational_scaffold",
    "boundary_condition": "none_or_declared_proxy",
    "gauge": "none_or_declared_proxy"
}

print("SCALES:", SCALES)

SCALES: [10, 50, 100, 500]


## 5. Simulation Construction with Asynchronous Configuration

Set up drive heterogeneity and randomized states to ensure an asynchronous-irregular regime.

In [3]:
def build_scale_configuration(n: int, seed: int):
    n_e = int(round(E_FRACTION * n))
    n_i = n - n_e
    
    # Preserve spatial density: extent increases proportionally to N
    extent_mm = float(n) * 0.01
    
    cfg = (jtfne.Configuration()
        .runtime(seed=seed, dtype=DTYPE)
        .column(name=f"scale_{n}", layers=LAYER_NAMES, n=n)
        .cell_types({"E": n_e, "PV": n_i})
        .connectivity(kind="dense_signed_ei_metadata")
        .emitter(family="izhikevich", preset="cortical_eig")
        .probes(PROBE_MODES))
    return cfg, extent_mm

def construct_and_simulate(cfg, extent_mm: float, seed: int):
    model = jtfne.construct(cfg)
    
    # Induce noisy asynchronous parameter modifications
    emitter_params = model.params["emitter"]
    n_neurons = emitter_params.n_neurons
    
    key = jax.random.PRNGKey(seed)
    k1, k2 = jax.random.split(key)
    
    # 1. Heterogeneous baseline drive per neuron
    drive_offset = jax.random.normal(k1, shape=(n_neurons,)) * 1.5
    new_drive = emitter_params.drive + drive_offset
    
    # 2. Randomized initial voltage/recovery states
    new_v0 = jax.random.uniform(k2, shape=(n_neurons,), minval=-70.0, maxval=-50.0)
    new_u0 = emitter_params.b * new_v0
    
    # 3. Weaker recurrent synchronizing weights
    new_W = emitter_params.W * 0.05
    
    updated_emitter = replace(
        emitter_params,
        drive=new_drive,
        v0=new_v0,
        u0=new_u0,
        W=new_W
    )
    model.params["emitter"] = updated_emitter
    
    # Run simulation
    signals = jtfne.simulate(model, duration_ms=DURATION_MS, dt_ms=DT_MS, seed=seed)
    return model, signals

## 6. Whole-Window Power Estimation and Metrics

Implement rFFT-based absolute power spectral density and log-log polyfit algorithms.

In [4]:
def compute_absolute_power(y: np.ndarray, dt_ms: float) -> tuple[np.ndarray, np.ndarray]:
    y = np.asarray(y, dtype=np.float64)
    y = y - np.mean(y)  # demean
    fs = 1000.0 / dt_ms
    window = np.hanning(y.size)
    yw = y * window
    scale = np.sum(window**2) * fs
    spec = np.abs(np.fft.rfft(yw)) ** 2 / max(scale, 1e-15)
    freq = np.fft.rfftfreq(y.size, d=1.0 / fs)
    return freq, spec

def fit_log_log_power_law(freq: np.ndarray, psd: np.ndarray) -> float:
    mask = (freq >= FIT_BAND[0]) & (freq <= FIT_BAND[1]) & (freq > 0)
    if not np.any(mask):
        return 0.0
    log_f = np.log10(freq[mask])
    log_p = np.log10(psd[mask])
    p = np.polyfit(log_f, log_p, 1)
    return float(-p[0])  # Exponent alpha is negative slope

def estimate_mean_rate_hz(spikes: np.ndarray, n: int) -> float:
    duration_s = DURATION_MS / 1000.0
    return float(np.sum(spikes > 0)) / (n * duration_s)

def synchrony_proxy_from_spikes(spikes: np.ndarray) -> float:
    bin_size = int(round(SYNCHRONY_BIN_MS / DT_MS))
    n_bins = spikes.shape[0] // bin_size
    if n_bins < 2:
        return 0.0
    trimmed = spikes[: n_bins * bin_size]
    binned_population_counts = trimmed.reshape(n_bins, bin_size, -1).sum(axis=(1, 2))
    mean_count = np.mean(binned_population_counts)
    if mean_count == 0:
        return 0.0
    return float(np.var(binned_population_counts) / mean_count)

## 7. Scaling Execution Loop

Execute simulation and analyze absolute spectra across sizes.

In [5]:
results = {}
for index, n in enumerate(SCALES):
    scale_seed = SEED + 1000 * index
    print(f"Executing N={n}, seed={scale_seed}")
    cfg, extent_mm = build_scale_configuration(n, scale_seed)
    model, signals = construct_and_simulate(cfg, extent_mm, scale_seed)
    
    spikes = np.asarray(signals.spikes)
    lfp = np.asarray(signals.field.lfp_proxy)
    y = lfp.mean(axis=1) if lfp.ndim == 2 else lfp
    
    freq, psd = compute_absolute_power(y, DT_MS)
    alpha = fit_log_log_power_law(freq, psd)
    
    mean_rate = estimate_mean_rate_hz(spikes, n)
    synchrony = synchrony_proxy_from_spikes(spikes)
    
    low_freq_mask = (freq >= 0.5) & (freq <= 4.0)
    low_freq_power = float(np.mean(psd[low_freq_mask]))
    
    results[n] = {
        "spikes": spikes,
        "freq": freq,
        "psd": psd,
        "alpha": alpha,
        "mean_rate": mean_rate,
        "synchrony": synchrony,
        "low_freq_power": low_freq_power,
        "extent_mm": extent_mm
    }
    print(f"  Rate: {mean_rate:.2f} Hz, Alpha: {alpha:.2f}, Synchrony: {synchrony:.2f}")

Executing N=10, seed=303


  Rate: 6.80 Hz, Alpha: 0.02, Synchrony: 1.79
Executing N=50, seed=1303


  Rate: 8.92 Hz, Alpha: -0.48, Synchrony: 3.27
Executing N=100, seed=2303


  Rate: 8.91 Hz, Alpha: -0.11, Synchrony: 5.87
Executing N=500, seed=3303


  Rate: 9.78 Hz, Alpha: 0.21, Synchrony: 27.14


## 8. Quantitative Visualizations

Plot separate panels for asynchronous rasters, absolute power spectra, power-law slope fits, and scale curves.

In [6]:
time_ms = np.arange(int(round(DURATION_MS / DT_MS))) * DT_MS

# 1. Asynchronous raster by scale
fig, axes = plt.subplots(len(SCALES), 1, figsize=(10, 8), sharex=True)
for idx, n in enumerate(SCALES):
    spk = results[n]["spikes"]
    ax = axes[idx]
    t_indices, n_indices = np.nonzero(spk)
    ax.scatter(t_indices * DT_MS, n_indices, s=1, color="black", alpha=0.6)
    ax.set_title(f"Scale N={n} (Asynchronous irregular texture)")
    ax.set_ylabel("Neuron index")
plt.xlabel("Time (ms)")
plt.tight_layout()
fig.savefig("01_suite03_raster_by_scale.png")
plt.close()

# 2. Absolute power spectrum by scale on log-log axes
fig, ax = plt.subplots(figsize=(8, 5))
for n in SCALES:
    ax.loglog(results[n]["freq"], results[n]["psd"], label=f"N={n}")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Absolute power spectral density")
ax.set_title("Absolute power spectrum P(f) by scale")
ax.legend()
plt.tight_layout()
fig.savefig("02_suite03_absolute_power_spectrum.png")
plt.close()

# 3. Log-log power-law fit overlay
fig, ax = plt.subplots(figsize=(8, 5))
n_target = SCALES[-1]
target_f = results[n_target]["freq"]
target_p = results[n_target]["psd"]
target_alpha = results[n_target]["alpha"]
ax.loglog(target_f, target_p, color="blue", label=f"N={n_target} spectrum")

mask = (target_f >= FIT_BAND[0]) & (target_f <= FIT_BAND[1]) & (target_f > 0)
fit_f = target_f[mask]
log_f = np.log10(fit_f)
log_p = np.log10(target_p[mask])
poly = np.polyfit(log_f, log_p, 1)
fit_y = 10 ** (poly[0] * log_f + poly[1])
ax.loglog(fit_f, fit_y, "r--", lw=2, label=f"Log-log power-law fit (alpha={target_alpha:.2f})")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Power")
ax.set_title(f"1/f^alpha scaling fit (N={n_target})")
ax.legend()
plt.tight_layout()
fig.savefig("03_suite03_log_log_power_law_fit.png")
plt.close()

# 4. Alpha vs N
fig, ax = plt.subplots(figsize=(6, 4))
alphas = [results[n]["alpha"] for n in SCALES]
ax.plot(SCALES, alphas, "o-", color="purple")
ax.set_xlabel("Population size N")
ax.set_ylabel("Fitted alpha")
ax.set_title("Alpha versus N")
plt.tight_layout()
fig.savefig("04_suite03_alpha_vs_scale.png")
plt.close()

# 5. Low-frequency absolute power vs N
fig, ax = plt.subplots(figsize=(6, 4))
low_powers = [results[n]["low_freq_power"] for n in SCALES]
ax.plot(SCALES, low_powers, "s-", color="green")
ax.set_xlabel("Population size N")
ax.set_ylabel("Low-frequency absolute power (0.5-4 Hz)")
ax.set_title("Low-frequency absolute power versus N")
plt.tight_layout()
fig.savefig("05_suite03_low_frequency_absolute_power_vs_scale.png")
plt.close()

# 6. Synchrony proxy vs N
fig, ax = plt.subplots(figsize=(6, 4))
synchs = [results[n]["synchrony"] for n in SCALES]
ax.plot(SCALES, synchs, "d-", color="red")
ax.set_xlabel("Population size N")
ax.set_ylabel("Synchrony proxy (Fano ratio)")
ax.set_title("Synchrony proxy versus N")
plt.tight_layout()
fig.savefig("06_suite03_synchrony_proxy_vs_scale.png")
plt.close()

print("All 6 figures generated cleanly.")

All 6 figures generated cleanly.


## 9. Manifest and Validation Export

Write validation receipt containing strict asynchronous irregular indicators, density, and slope exponents.

In [7]:
receipt = {
    "neurons_per_declared_extent": 100.0,  # Constant spatiotemporal density ratio
    "duration_ms": DURATION_MS,
    "dt_ms": DT_MS,
    "mean_rate_by_scale": {str(n): results[n]["mean_rate"] for n in SCALES},
    "alpha_by_scale": {str(n): results[n]["alpha"] for n in SCALES},
    "low_freq_power_by_scale": {str(n): results[n]["low_freq_power"] for n in SCALES},
    "synchrony_by_scale": {str(n): results[n]["synchrony"] for n in SCALES},
    "synchrony_regime": "asynchronous_irregular_proxy",
    "global_burst_dominated": False,
    "non_negotiable_boundaries": {
        "physical_amplitude_claim_allowed": False,
        "field_solver_status": "laminar_proxy_no_pde",
        "science_mode": "truth_safe_unverified"
    }
}

with open("suite_no3_v0310_execution_receipt.json", "w") as f:
    json.dump(receipt, f, indent=2, allow_nan=False)

print("Receipt manifest exported successfully.")

Receipt manifest exported successfully.


## 10. Scope Boundaries

### Scope boundary

This is a proxy source-to-readout projection unless a run supplies physical geometry, calibrated source units, conductivity, boundary conditions, gauge handling, a physical field solver, and validation evidence.

### What This Tutorial Is
✓ A scale-dependent absolute power scaling tutorial  
✓ A demonstration of noisy asynchronous irregular regimes  
✓ JAX-based, deterministic, and reproducible  

### What This Tutorial Is NOT
✗ A biophysically validated dataset  
✗ A calibrated physical field solver  

### Scope Metadata
```json
{
  "scope_status": "computational_scaffold",
  "calibration_status": "uncalibrated_phenomenological",
  "readout_status": "proxy_scale",
  "field_mode": "proxy_convolution_no_pde",
  "physical_amplitude_claim_allowed": false
}
```